<a href="https://colab.research.google.com/github/VIKAS-PURBIA/E-Commerce-SQL-Data-Analysis/blob/main/Task_3_Ecommerce_SQL_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce SQL Data Analysis — Task 3
**Elevate Labs | SQLite | Complete end-to-end project**

This notebook completes the practical workflow for the Task 3 brief:
1. Load dataset and create SQLite database
2. Explore schema and relationships
3. Check data quality (missing values, duplicates, key integrity)
4. Perform SQL analysis with filters, sorting, grouping, joins, subqueries and aggregates
5. Create an analysis view and indexes; inspect query plan
6. Export results and prepare submission files

**Dataset note:** The included practice dataset is synthetic and contains no real customer data.

## Step 1 — Setup and load the dataset

In [1]:
import sqlite3, pandas as pd, numpy as np, os, zipfile, glob, re
from pathlib import Path
from IPython.display import display

print("SQLite:", sqlite3.sqlite_version)
print("Pandas:", pd.__version__)

SQLite: 3.45.1
Pandas: 2.2.3


### Upload the provided dataset ZIP in Google Colab
Upload `Ecommerce_SQL_Practice_Dataset.zip` when prompted. If running locally, set `DATASET_DIR` to the folder containing the CSV files.

In [3]:
DATASET_DIR = None  # For local Jupyter, set e.g. "dataset"

try:
    from google.colab import files
    uploaded = files.upload()
    zip_files = [f for f in uploaded if f.lower().endswith(".zip")]
    if zip_files:
        os.makedirs("/content/ecommerce_dataset", exist_ok=True)
        with zipfile.ZipFile(zip_files[0]) as zf:
            zf.extractall("/content/ecommerce_dataset")
        DATASET_DIR = "/content/ecommerce_dataset"
except ImportError:
    print("Not running in Google Colab. Using DATASET_DIR or ./dataset.")

if DATASET_DIR is None:
    DATASET_DIR = "dataset"

csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {DATASET_DIR}. Upload the dataset ZIP in Colab "
        "or set DATASET_DIR to the folder containing the CSV files."
    )
print("CSV files found:", [Path(f).name for f in csv_files])

Saving Ecommerce_SQL_Practice_Dataset.zip to Ecommerce_SQL_Practice_Dataset.zip
CSV files found: ['orders.csv', 'payments.csv', 'products.csv', 'customers.csv', 'order_items.csv']


In [4]:
DB_PATH = "ecommerce.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
for file in csv_files:
    df = pd.read_csv(file)
    table = re.sub(r"[^a-zA-Z0-9_]", "_", Path(file).stem).lower()
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"{table}: {len(df):,} rows, {len(df.columns)} columns")

conn.execute("PRAGMA foreign_keys = ON")
print("SQLite database created:", DB_PATH)

orders: 220 rows, 4 columns
payments: 199 rows, 6 columns
products: 25 rows, 5 columns
customers: 80 rows, 5 columns
order_items: 536 rows, 5 columns
SQLite database created: ecommerce.db


## Step 2 — Explore tables, columns, and relationships

In [5]:
tables = pd.read_sql_query(
    "SELECT name AS table_name FROM sqlite_master "
    "WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name", conn
)
display(tables)

for table in tables["table_name"]:
    safe = table.replace('"', '""')
    count = pd.read_sql_query(f'SELECT COUNT(*) AS rows FROM "{safe}"', conn).iloc[0, 0]
    print(f"\n{table}: {count:,} rows")
    display(pd.read_sql_query(f'PRAGMA table_info("{safe}")', conn)[["name", "type", "pk"]])
    display(pd.read_sql_query(f'SELECT * FROM "{safe}" LIMIT 5', conn))

,table_name
0,customers
1,order_items
2,orders
3,payments
4,products



customers: 80 rows


,name,type,pk
0,customer_id,INTEGER,0
1,customer_name,TEXT,0
2,email,TEXT,0
3,city,TEXT,0
4,signup_date,TEXT,0


,customer_id,customer_name,email,city,signup_date
0,1,Customer 001,customer001@example.com,Jaipur,2025-10-20
1,2,Customer 002,customer002@example.com,Kolkata,2024-11-02
2,3,Customer 003,customer003@example.com,Hyderabad,2024-04-22
3,4,Customer 004,customer004@example.com,Bengaluru,2025-08-05
4,5,Customer 005,customer005@example.com,Bengaluru,2025-03-16



order_items: 536 rows


,name,type,pk
0,order_item_id,INTEGER,0
1,order_id,INTEGER,0
2,product_id,INTEGER,0
3,quantity,INTEGER,0
4,unit_price,INTEGER,0


... (6 outputs left out)

,payment_id,order_id,payment_date,payment_method,payment_amount,payment_status
0,1,1001,2025-05-14,Credit Card,2598,Paid
1,2,1002,2025-02-19,Net Banking,3894,Paid
2,3,1003,2025-04-25,UPI,6295,Paid
3,4,1004,2025-03-13,Net Banking,14891,Paid
4,5,1005,2025-04-01,Cash on Delivery,1798,Paid



products: 25 rows


,name,type,pk
0,product_id,INTEGER,0
1,product_name,TEXT,0
2,category,TEXT,0
3,unit_price,INTEGER,0
4,stock_quantity,INTEGER,0


,product_id,product_name,category,unit_price,stock_quantity
0,101,Wireless Mouse,Electronics,799,149
1,102,Bluetooth Headphones,Electronics,1899,101
2,103,USB-C Cable,Electronics,299,64
3,104,Laptop Stand,Electronics,1199,63
4,105,Mechanical Keyboard,Electronics,3499,65


### Relationship map

- `customers.customer_id` → `orders.customer_id`
- `orders.order_id` → `order_items.order_id`
- `products.product_id` → `order_items.product_id`
- `orders.order_id` → `payments.order_id`

In [6]:
# Check orphaned foreign-key references
integrity_checks = {
    "orders_without_customer": """
        SELECT COUNT(*) AS issue_count FROM orders o
        LEFT JOIN customers c ON c.customer_id=o.customer_id
        WHERE c.customer_id IS NULL
    """,
    "items_without_order": """
        SELECT COUNT(*) AS issue_count FROM order_items i
        LEFT JOIN orders o ON o.order_id=i.order_id
        WHERE o.order_id IS NULL
    """,
    "items_without_product": """
        SELECT COUNT(*) AS issue_count FROM order_items i
        LEFT JOIN products p ON p.product_id=i.product_id
        WHERE p.product_id IS NULL
    """
}
for label, query in integrity_checks.items():
    value = pd.read_sql_query(query, conn).iloc[0, 0]
    print(label, "=", value)

orders_without_customer = 0
items_without_order = 0
items_without_product = 0


## Step 3 — Data quality checks

In [7]:
# Missing-value counts for every column in every table
for table in tables["table_name"]:
    safe = table.replace('"', '""')
    cols = pd.read_sql_query(f'PRAGMA table_info("{safe}")', conn)["name"].tolist()
    print(f"\nMissing values in {table}:")
    if not cols:
        continue
    expressions = ", ".join(
        f'SUM(CASE WHEN "{c.replace(chr(34), chr(34)*2)}" IS NULL THEN 1 ELSE 0 END) AS "{c}"'
        for c in cols
    )
    display(pd.read_sql_query(f'SELECT {expressions} FROM "{safe}"', conn))


Missing values in customers:


,customer_id,customer_name,email,city,signup_date
0,0,0,0,0,0



Missing values in order_items:


,order_item_id,order_id,product_id,quantity,unit_price
0,0,0,0,0,0



Missing values in orders:


,order_id,customer_id,order_date,order_status
0,0,0,0,0



Missing values in payments:


,payment_id,order_id,payment_date,payment_method,payment_amount,payment_status
0,0,0,0,0,0,0



Missing values in products:


,product_id,product_name,category,unit_price,stock_quantity
0,0,0,0,0,0


In [8]:
# Duplicate checks for expected primary keys
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id"
}
for table, key in primary_keys.items():
    query = f'''
        SELECT "{key}" AS key_value, COUNT(*) AS occurrences
        FROM "{table}"
        GROUP BY "{key}"
        HAVING COUNT(*) > 1
    '''
    duplicates = pd.read_sql_query(query, conn)
    print(f"{table}.{key} duplicate key rows:", len(duplicates))
    if not duplicates.empty:
        display(duplicates)

customers.customer_id duplicate key rows: 0
products.product_id duplicate key rows: 0
orders.order_id duplicate key rows: 0
order_items.order_item_id duplicate key rows: 0
payments.payment_id duplicate key rows: 0


## Step 4 — SQL analysis queries

### 4.1 SELECT, WHERE, ORDER BY

In [9]:
q1 = """
SELECT order_id, customer_id, order_date, order_status
FROM orders
WHERE order_status = 'Completed'
ORDER BY order_date DESC
LIMIT 10;
"""
display(pd.read_sql_query(q1, conn))

,order_id,customer_id,order_date,order_status
0,1162,19,2025-10-25,Completed
1,1171,30,2025-10-25,Completed
2,1015,41,2025-10-16,Completed
3,1010,2,2025-10-15,Completed
4,1198,60,2025-10-12,Completed
5,1197,63,2025-10-09,Completed
6,1163,20,2025-10-08,Completed
7,1079,64,2025-10-05,Completed
8,1017,56,2025-09-30,Completed
9,1160,72,2025-09-29,Completed


### 4.2 GROUP BY and HAVING — revenue by customer

In [10]:
q2 = """
SELECT c.customer_id, c.customer_name,
       SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.customer_name
HAVING SUM(oi.quantity * oi.unit_price) > 10000
ORDER BY total_revenue DESC;
"""
display(pd.read_sql_query(q2, conn))

,customer_id,customer_name,total_revenue
0,75,Customer 075,72158
1,34,Customer 034,40166
2,67,Customer 067,36119
3,43,Customer 043,35577
4,65,Customer 065,34678
5,19,Customer 019,33177
6,12,Customer 012,30924
7,9,Customer 009,29180
8,14,Customer 014,28023
9,35,Customer 035,27078


### 4.3 INNER JOIN — customer orders

In [11]:
q3 = """
SELECT c.customer_name, o.order_id, o.order_date, o.order_status
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY o.order_date DESC
LIMIT 15;
"""
display(pd.read_sql_query(q3, conn))

,customer_name,order_id,order_date,order_status
0,Customer 035,1216,2025-10-27,Pending
1,Customer 019,1162,2025-10-25,Completed
2,Customer 030,1171,2025-10-25,Completed
3,Customer 038,1080,2025-10-24,Shipped
4,Customer 067,1140,2025-10-20,Shipped
5,Customer 034,1219,2025-10-19,Shipped
6,Customer 001,1170,2025-10-18,Shipped
7,Customer 065,1023,2025-10-18,Shipped
8,Customer 041,1015,2025-10-16,Completed
9,Customer 059,1152,2025-10-16,Shipped


### 4.4 LEFT JOIN — customers with and without orders

In [12]:
q4 = """
SELECT c.customer_id, c.customer_name,
       COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY order_count, c.customer_id;
"""
display(pd.read_sql_query(q4, conn).head(15))

,customer_id,customer_name,order_count
0,17,Customer 017,0
1,18,Customer 018,0
2,28,Customer 028,0
3,36,Customer 036,0
4,3,Customer 003,1
5,6,Customer 006,1
6,21,Customer 021,1
7,27,Customer 027,1
8,29,Customer 029,1
9,31,Customer 031,1


### 4.5 Aggregates — revenue by product category

In [13]:
q5 = """
SELECT p.category,
       SUM(oi.quantity) AS units_sold,
       SUM(oi.quantity * oi.unit_price) AS sales_revenue,
       ROUND(AVG(oi.unit_price), 2) AS average_unit_price
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY sales_revenue DESC;
"""
display(pd.read_sql_query(q5, conn))

,category,units_sold,sales_revenue,average_unit_price
0,Clothing,208,341292,1699.00
1,Sports,226,280074,1261.39
2,Electronics,180,257620,1459.42
3,Home & Kitchen,199,220901,1080.00
4,Books,231,159769,686.73


### 4.6 Subquery — customers spending above average

In [14]:
q6 = """
WITH customer_spend AS (
    SELECT c.customer_id, c.customer_name,
           SUM(oi.quantity * oi.unit_price) AS total_spend
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT *
FROM customer_spend
WHERE total_spend > (SELECT AVG(total_spend) FROM customer_spend)
ORDER BY total_spend DESC;
"""
display(pd.read_sql_query(q6, conn))

,customer_id,customer_name,total_spend
0,75,Customer 075,72158
1,34,Customer 034,40166
2,67,Customer 067,36119
3,43,Customer 043,35577
4,65,Customer 065,34678
5,19,Customer 019,33177
6,12,Customer 012,30924
7,9,Customer 009,29180
8,14,Customer 014,28023
9,35,Customer 035,27078


### 4.7 SUM, AVG, COUNT and average revenue per user

In [15]:
q7 = """
SELECT SUM(quantity * unit_price) AS gross_revenue,
       ROUND(AVG(quantity * unit_price), 2) AS average_line_item_value,
       COUNT(DISTINCT order_id) AS orders_with_items
FROM order_items;
"""
display(pd.read_sql_query(q7, conn))

# ARPU uses all registered customers as denominator
q_arpu = """
WITH customer_revenue AS (
    SELECT o.customer_id, SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id
)
SELECT ROUND(COALESCE(SUM(revenue), 0) * 1.0 /
             (SELECT COUNT(*) FROM customers), 2) AS average_revenue_per_customer
FROM customer_revenue;
"""
display(pd.read_sql_query(q_arpu, conn))

,gross_revenue,average_line_item_value,orders_with_items
0,1259656,2350.1,220


,average_revenue_per_customer
0,15745.7


### 4.8 NULL handling and payment coverage

In [16]:
q8 = """
SELECT o.order_id, o.order_status,
       COALESCE(p.payment_status, 'No payment record') AS payment_status,
       COALESCE(p.payment_amount, 0) AS payment_amount
FROM orders o
LEFT JOIN payments p ON o.order_id = p.order_id
ORDER BY o.order_id
LIMIT 20;
"""
display(pd.read_sql_query(q8, conn))

,order_id,order_status,payment_status,payment_amount
0,1001,Completed,Paid,2598
1,1002,Completed,Paid,3894
2,1003,Completed,Paid,6295
3,1004,Completed,Paid,14891
4,1005,Shipped,Paid,1798
5,1006,Completed,Refunded,6243
6,1007,Cancelled,Paid,4098
7,1008,Cancelled,Paid,4797
8,1009,Completed,Paid,599
9,1010,Completed,Paid,2499


## Step 5 — Create a view and optimize with indexes

In [17]:
conn.executescript("""
DROP VIEW IF EXISTS vw_order_revenue;
CREATE VIEW vw_order_revenue AS
SELECT o.order_id, o.customer_id, o.order_date, o.order_status,
       SUM(oi.quantity * oi.unit_price) AS order_revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.order_id, o.customer_id, o.order_date, o.order_status;
""")
display(pd.read_sql_query(
    "SELECT * FROM vw_order_revenue ORDER BY order_revenue DESC LIMIT 10", conn
))

,order_id,customer_id,order_date,order_status,order_revenue
0,1154,64,2025-03-02,Shipped,19292
1,1088,61,2025-02-15,Completed,16489
2,1171,30,2025-10-25,Completed,15243
3,1095,35,2025-01-25,Pending,14992
4,1004,75,2025-03-13,Completed,14891
5,1156,43,2025-07-10,Completed,14694
6,1054,68,2025-04-03,Pending,14191
7,1145,55,2025-06-16,Completed,14191
8,1097,53,2025-05-26,Returned,13894
9,1016,75,2025-04-20,Shipped,13793


In [18]:
# Index columns commonly used in joins and filters
index_sql = """
CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id);
CREATE INDEX IF NOT EXISTS idx_orders_order_date ON orders(order_date);
CREATE INDEX IF NOT EXISTS idx_order_items_order_id ON order_items(order_id);
CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON order_items(product_id);
CREATE INDEX IF NOT EXISTS idx_payments_order_id ON payments(order_id);
"""
conn.executescript(index_sql)
conn.commit()

# Inspect query plan. On a small dataset SQLite may still choose a table scan.
plan = pd.read_sql_query(
    "EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 10", conn
)
display(plan)

print("Indexes:")
display(pd.read_sql_query(
    "SELECT name, tbl_name FROM sqlite_master WHERE type='index' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    conn
))

,id,parent,notused,detail
0,3,0,0,SEARCH orders USING INDEX idx_orders_customer_...


Indexes:


,name,tbl_name
0,idx_order_items_order_id,order_items
1,idx_order_items_product_id,order_items
2,idx_orders_customer_id,orders
3,idx_orders_order_date,orders
4,idx_payments_order_id,payments


## Step 6 — Export query results and database

In [19]:
# Export key analysis outputs as CSV files
os.makedirs("outputs", exist_ok=True)
export_queries = {
    "revenue_by_category": q5,
    "top_customers_above_average": q6,
    "order_revenue": "SELECT * FROM vw_order_revenue ORDER BY order_revenue DESC",
    "payment_coverage": q8,
    "average_revenue_per_customer": q_arpu
}
for filename, query in export_queries.items():
    result = pd.read_sql_query(query, conn)
    result.to_csv(f"outputs/{filename}.csv", index=False)
    print(f"Saved outputs/{filename}.csv ({len(result)} rows)")

conn.commit()
print("SQLite database:", os.path.abspath(DB_PATH))
print("Database size (KB):", round(os.path.getsize(DB_PATH)/1024, 2))

Saved outputs/revenue_by_category.csv (5 rows)
Saved outputs/top_customers_above_average.csv (34 rows)
Saved outputs/order_revenue.csv (220 rows)
Saved outputs/payment_coverage.csv (20 rows)
Saved outputs/average_revenue_per_customer.csv (1 rows)
SQLite database: /content/ecommerce.db
Database size (KB): 100.0


## Task 3 completion checklist

- [ ] Dataset loaded into SQLite
- [ ] Tables, columns, and relationships inspected
- [ ] Missing values, duplicates, and orphan records checked
- [ ] SELECT, WHERE, ORDER BY, GROUP BY, HAVING completed
- [ ] INNER JOIN and LEFT JOIN completed
- [ ] Subquery and aggregate functions completed
- [ ] View created
- [ ] Indexes created and query plan inspected
- [ ] SQL file, database, outputs, screenshots, and README prepared

**Submission reminder:** The brief requests a SQL file plus screenshots of outputs and a GitHub repository with code, dataset, screenshots, and a short README. See `README.md` and `Interview_Questions.md` in the project folder.